In [1]:
from IPython.display import IFrame

IFrame("FD_five_pt_stencil.pdf", width=1000, height=800)

## Answer - 7a

In [2]:
import numpy as np

# Matrix D for the internal node of dimension N\times N for uniform grid
def matrix_D(M):
    N = M-1
    D = np.zeros((N,N))
    for i in range(N):
        D[i, i] = 4
        if i != 0:
            D[i, i-1] = -1
        if i != N-1:
            D[i, i+1] = -1
    return D

# Matrix of the formulation [[0, -1], [-1, 0]]
def create_tridiagonal_matrix(M):
    N = M-1
    A = np.zeros((N, N))
    for i in range(N):
        if i != 0:
            A[i, i-1] = -1
        if i != N-1:
            A[i, i+1] = -1
    return A

# Right hand side
def f_rhs(x, y):
    return -(12*x**2*y**5 + 20*x**4*y**3 + 17*np.sin(x*y)*(x**2 + y**2))
    #return -4

# Exact Solution
def exact_solution(x, y):
    return x**4*y**5-17*np.sin(x*y)
    #return x**2+y**2

# Solve the Poisson Problem
def solve_poisson_rectangular_uniform_grid(M):
    # Interval points
    ax, bx = 0, 2
    ay, by = 0, 1
    h = (bx - ax) / M
    Nx = M - 1            # number of interior points for fixed-j in x-direction
    Ny = int((by - ay)/h) - 1  # number of interior points for fixed-i in y-direction

    D = matrix_D(Ny+1)
    I1 = np.eye(Nx)
    I2 = np.eye(Ny)
    # Diagonal part of matrix A
    A1 = np.kron(I1, D)
    # Other part of matrix A
    A2 = np.kron(create_tridiagonal_matrix(Nx+1), I2)
    A = (A1 + A2)/(h**2)
    # Load vector and Dirichlet BCs
    F = np.zeros(Nx * Ny)
    for i in range(Nx):
        for j in range(Ny):
            x = (i+1)*h
            y = (j+1)*h
            l = i*Ny + j
            F[l] = f_rhs(x, y)

            # Boundary contributions
            if i == 0:
                F[l] += exact_solution(ax, y) / h**2
            if i == Nx - 1:
                F[l] += exact_solution(bx, y) / h**2
            if j == 0:
                F[l] += exact_solution(x, ay) / h**2
            if j == Ny - 1:
                F[l] += exact_solution(x, by) / h**2

    # Solve system
    U_vec = np.linalg.solve(A, F)

    # Reshape to 2D grid
    U_grid = U_vec.reshape((Nx, Ny))

    # Exact solution
    U_exact = np.zeros((Nx, Ny))
    for i in range(Nx):
        for j in range(Ny):
            x = (i+1)*h
            y = (j+1)*h
            U_exact[i, j] = exact_solution(x, y)

    # Infinity norm of error
    error_inf = np.max(np.abs(U_grid - U_exact))

    return h, error_inf

# Mesh sizes to test
mesh_sizes = [4, 8, 16, 32, 64, 128]
error = []
step_size = []

for M in mesh_sizes:
    h, err_inf= solve_poisson_rectangular_uniform_grid(M)
    step_size.append(h)
    error.append(err_inf)

for i in range(len(step_size)):
    if i == 0:
        eoc = 0
    else:
        eoc = np.log(error[i-1] / error[i]) / np.log(step_size[i-1] / step_size[i])
    print(f"h : {step_size[i]:.5f}, Error : {error[i]:.7f}, EOC : {eoc:.5f}")

h : 0.50000, Error : 0.3632489, EOC : 0.00000
h : 0.25000, Error : 0.1080426, EOC : 1.74936
h : 0.12500, Error : 0.0319699, EOC : 1.75681
h : 0.06250, Error : 0.0081049, EOC : 1.97985
h : 0.03125, Error : 0.0020374, EOC : 1.99207
h : 0.01562, Error : 0.0005103, EOC : 1.99733


## Answer - 7b

In [3]:
# Matrix D for the internal node of dimension N\times N for non-uniform grid
def matrix_D_dash(M, hx, hy):
    N = M-1
    D = np.zeros((N,N))
    for i in range(N):
        D[i, i] = 2*( 1/hx**2 + 1/hy**2 )
        if i != 0:
            D[i, i-1] = -1/hy**2
        if i != N-1:
            D[i, i+1] = -1/hy**2
    return D


# Solve the Poisson Problem
def solve_poisson_rectangular_non_uniform_grid(M):
    # Interval points
    ax, bx = 0, 2
    ay, by = 0, 1
    hx = (bx - ax) / M[0]
    hy = (by - ay) / M[1]
    Nx = M[0] - 1              # number of interior points for fixed-j in x-direction
    Ny = M[1] - 1              # number of interior points for fixed-i in y-direction

    D = matrix_D_dash(Ny+1, hx, hy)
    I1 = np.eye(Nx)
    I2 = (1/hx**2)*np.eye(Ny)
    # Diagonal part of matrix A
    A1 = np.kron(I1, D)
    # Other part of matrix A
    A2 = np.kron(create_tridiagonal_matrix(Nx+1), I2)
    A = A1 + A2
    # Load vector and Dirichlet BCs
    F = np.zeros(Nx * Ny)
    for i in range(Nx):
        for j in range(Ny):
            x = (i+1)*hx
            y = (j+1)*hy
            l = i*Ny + j
            F[l] = f_rhs(x, y)

            # Boundary contributions
            if i == 0:
                F[l] += exact_solution(ax, y) / hx**2
            if i == Nx - 1:
                F[l] += exact_solution(bx, y) / hx**2
            if j == 0:
                F[l] += exact_solution(x, ay) / hy**2
            if j == Ny - 1:
                F[l] += exact_solution(x, by) / hy**2

    # Solve system
    U_vec = np.linalg.solve(A, F)

    # Reshape to 2D grid
    U_grid = U_vec.reshape((Nx, Ny))

    # Exact solution
    U_exact = np.zeros((Nx, Ny))
    for i in range(Nx):
        for j in range(Ny):
            x = (i+1)*hx
            y = (j+1)*hy
            U_exact[i, j] = exact_solution(x, y)

    # Infinity norm of error
    error_inf = np.max(np.abs(U_grid - U_exact))

    return hx, hy, error_inf

# Each entry is mesh size in x and y direction
mesh_sizes = [[5,10], [10,15], [20,33], [50,40], [100,77]]

for M in mesh_sizes:
    h_x, h_y, err_inf= solve_poisson_rectangular_non_uniform_grid(M)
    print(f"hx : {h_x:.5f}, hy : {h_y:.5f}, Error : {err_inf:.7f}")

hx : 0.40000, hy : 0.10000, Error : 0.0189038
hx : 0.20000, hy : 0.06667, Error : 0.0090111
hx : 0.10000, hy : 0.03030, Error : 0.0019211
hx : 0.04000, hy : 0.02500, Error : 0.0013069
hx : 0.02000, hy : 0.01299, Error : 0.0003532


## Answer - 7c

In [4]:
# Solve the Poisson Problem
def solve_poisson_non_rectangular_non_uniform_grid(M):
    # Interval points
    ax, bx = 0, 1
    ay, by = 0, 1
    hx = (bx - ax) / M[0]
    hy = (by - ay) / M[1]
    Nx = M[0] - 1              # number of interior points for fixed-j in x-direction
    Ny = M[1] - 1              # number of interior points for fixed-i in y-direction

    ax1, ay1 = 0.5, 0.5
    nx = int((ax1-ax)/hx) - 1
    ny = int((by-ay1)/hy) - 1

    D = matrix_D_dash(Ny+1, hx, hy)
    I1 = np.eye(Nx)
    I2 = (1/hx**2)*np.eye(Ny)
    # Diagonal part of matrix A
    A1 = np.kron(I1, D)
    # Other part of matrix A
    A2 = np.kron(create_tridiagonal_matrix(Nx+1), I2)
    A = A1 + A2
    # Load vector and Dirichlet BCs
    F = np.zeros(Nx * Ny)
    for i in range(Nx):
        for j in range(Ny):
            x = (i+1)*hx
            y = (j+1)*hy
            if x<ax1 or (x>ax1 and y<ay1):
                l = i*Ny + j
                F[l] = f_rhs(x, y)
    
                # Boundary contributions
                if i == 0:
                    F[l] += exact_solution(ax, y) / hx**2
                if i == nx:
                    F[l] += exact_solution(ax1, y) / hx**2
                if i == Nx - 1:
                    F[l] += exact_solution(bx, y) / hx**2
                if j == 0:
                    F[l] += exact_solution(x, ay) / hy**2
                if j == ny:
                    F[l] += exact_solution(x, ay1) / hy**2
                if j == Ny - 1:
                    F[l] += exact_solution(x, by) / hy**2

    # Solve system
    U_vec = np.linalg.solve(A, F)

    # Reshape to 2D grid
    U_grid = U_vec.reshape((Nx, Ny))

    # Exact solution
    U_exact = np.zeros((Nx, Ny))
    for i in range(Nx):
        for j in range(Ny):
            x = (i+1)*hx
            y = (j+1)*hy
            if x<ax1 or (x>ax1 and y<ay1):
                U_exact[i, j] = exact_solution(x, y)

    # Infinity norm of error
    error_inf = np.max(np.abs(U_grid - U_exact))

    return hx, hy, error_inf

# Each entry is mesh size in x and y direction
mesh_sizes = [[5,10], [10,15], [20,33], [50,40], [100,77]]

for M in mesh_sizes:
    h_x, h_y, err_inf= solve_poisson_non_rectangular_non_uniform_grid(M)
    print(f"hx : {h_x:.5f}, hy : {h_y:.5f}, Error : {err_inf:.7f}")

hx : 0.20000, hy : 0.10000, Error : 4.6414360
hx : 0.10000, hy : 0.06667, Error : 9.8340145
hx : 0.05000, hy : 0.03030, Error : 24.0201574
hx : 0.02000, hy : 0.02500, Error : 11.0888823
hx : 0.01000, hy : 0.01299, Error : 59.6850318
